In [2]:
!pip install -q -U accelerate peft trl bitsandbytes

In [3]:
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
import os, gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, Trainer
from trl import SFTTrainer, SFTConfig, DPOTrainer
from peft import prepare_model_for_kbit_training, get_peft_model, LoraConfig
import torch


MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
CACHE_DIR = "./cache"

In [5]:
from datasets import load_dataset

SEED = 42
TRAIN_SUBSET = 4000
TEST_SUBSET = 800


train_raw = load_dataset(
    "HuggingFaceH4/ultrafeedback_binarized",
    split="train_prefs"
)

test_raw = load_dataset(
    "HuggingFaceH4/ultrafeedback_binarized",
    split="test_prefs"
)


train_raw = train_raw.shuffle(seed=SEED).select(range(TRAIN_SUBSET))
test_raw = test_raw.shuffle(seed=SEED).select(range(TEST_SUBSET))


train_sft_ds = train_raw.map(
    lambda x: {
        "messages": x["messages"]
    },
    remove_columns=train_raw.column_names
)

test_sft_ds = test_raw.map(
    lambda x: {
        "messages": x["messages"]
    },
    remove_columns=test_raw.column_names
)


train_dpo_ds = train_raw.map(
    lambda x: {
        "chosen": x["chosen"],
        "rejected": x["rejected"]
    },
    remove_columns=train_raw.column_names
)

test_dpo_ds = test_raw.map(
    lambda x: {
        "chosen": x["chosen"],
        "rejected": x["rejected"]
    },
    remove_columns=test_raw.column_names
)


In [32]:
train_dpo_ds[0]

{'chosen': [{'content': 'Do you know something about crystallography and structure factor?',
   'role': 'user'},
  {'content': 'Crystallography is the science of the arrangement of atoms in solids. It is a vast and interdisciplinary field that has applications in physics, chemistry, materials science, biology, and engineering.\n\nThe structure factor is a mathematical function that is used to describe the diffraction of waves by a crystal. It is a complex number that is related to the atomic positions in the crystal.\n\nThe structure factor can be used to calculate the intensity of the diffracted waves. This information can be used to determine the atomic positions in the crystal and to study the structure of materials.\n\nCrystallography is a powerful tool for understanding the structure of materials. It has been used to determine the structures of many important materials, including metals, semiconductors, and pharmaceuticals. It is also used to study the structure of biological mate

Test Tokenizer

In [33]:
tk = AutoTokenizer.from_pretrained(MODEL_ID)
text = tk.apply_chat_template(train_sft_ds[0]['messages'], tokenize=False, add_generation_prompt=False)

text

'<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nDo you know something about crystallography and structure factor?<|im_end|>\n<|im_start|>assistant\nCrystallography is the science of the arrangement of atoms in solids. It is a vast and interdisciplinary field that has applications in physics, chemistry, materials science, biology, and engineering.\n\nThe structure factor is a mathematical function that is used to describe the diffraction of waves by a crystal. It is a complex number that is related to the atomic positions in the crystal.\n\nThe structure factor can be used to calculate the intensity of the diffracted waves. This information can be used to determine the atomic positions in the crystal and to study the structure of materials.\n\nCrystallography is a powerful tool for understanding the structure of materials. It has been used to determine the structures of many important materials, including metals, sem

LoRA and training args

In [48]:
args = dict(
    learning_rate=2e-4,
    num_train_epochs=1,
    max_steps=200,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    bf16=False,
    fp16=True,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=20,

    lr_scheduler_type="cosine",
    warmup_steps=5,

    max_length=512,

    gradient_checkpointing=True,

    seed=42,
    report_to="none",
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

SFT

In [15]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [49]:
def params_stats(model):
  total = sum(p.numel() for p in model.parameters())
  trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
  return {"total": total, "trainable":trainable, "ratio_pct": trainable * 100/total}

def apply_chat(e, tok):
    return tok.apply_chat_template(
        e["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

def LoRA(output_dir='trl_lora'):
  gc.collect()
  torch.cuda.empty_cache()
  tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
  if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
  tokenizer.padding_side = "right"
  model = AutoModelForCausalLM.from_pretrained(
      MODEL_ID,
      torch_dtype=torch.float16,
      device_map="auto",
      cache_dir=CACHE_DIR
  )
  model.config.use_cache=False

  model = get_peft_model(model, peft_config)
  ps = params_stats(model)
  print(f"trainable={ps['trainable']:,} ({ps['ratio_pct']:.3f}%)")

  config = SFTConfig(
      output_dir=output_dir,
      optim="adamw_torch",
      **args
  )

  trainer = SFTTrainer(
    model=model,
    args=config,
    train_dataset=train_sft_ds,
    eval_dataset=test_sft_ds,
    processing_class=tokenizer,
    formatting_func=lambda ex: apply_chat(ex, tokenizer),
  )

  trainer.train()

  adapter_dir = os.path.join(output_dir, "adapter")
  model.save_pretrained(adapter_dir)
  tokenizer.save_pretrained(adapter_dir)

LoRA()


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable=4,358,144 (0.282%)


Tokenizing train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
20,1.486664,1.341233
40,1.315577,1.272800
60,1.333632,1.260715
80,1.315007,1.253045
100,1.305500,1.249957
120,1.269031,1.246563
140,1.345956,1.244576
160,1.321059,1.243341
180,1.286100,1.242807
200,1.329466,1.242704


DPO

In [8]:
from trl import DPOTrainer, DPOConfig
from peft import PeftModel

def format_dpo(example, tokenizer):
    prompt = tokenizer.apply_chat_template(
        example["chosen"][:-1],
        tokenize=False,
        add_generation_prompt=True,
    )
    chosen = example["chosen"][-1]["content"]
    rejected = example["rejected"][-1]["content"]
    return {
        "prompt": prompt,
        "chosen": chosen,
        "rejected": rejected,
    }

def train_DPO(
    sft_adapter_dir="trl_lora/adapter",
    output_dir="trl_dpo"
):
    gc.collect()
    torch.cuda.empty_cache()

    tokenizer = AutoTokenizer.from_pretrained(sft_adapter_dir)
    tokenizer.padding_side = "right"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
        cache_dir=CACHE_DIR,
    )
    base_model.config.use_cache = False

    model = PeftModel.from_pretrained(
        base_model,
        sft_adapter_dir,
        is_trainable=True,
    )


    train_ds = train_dpo_ds.map(
        lambda x: format_dpo(x, tokenizer),
        remove_columns=train_dpo_ds.column_names,
    )
    test_ds = test_dpo_ds.map(
        lambda x: format_dpo(x, tokenizer),
        remove_columns=test_dpo_ds.column_names,
    )

    dpo_args = DPOConfig(
        output_dir=output_dir,
        learning_rate=5e-5,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=1,
        max_steps=200,
        logging_steps=20,
        eval_strategy="steps",
        eval_steps=20,
        save_steps=100,
        warmup_steps=5,
        lr_scheduler_type="cosine",
        fp16=True,
        bf16=False,
        gradient_checkpointing=True,
        report_to="none",
        beta=0.1,
        max_length=640
    )

    trainer = DPOTrainer(
        model=model,
        args=dpo_args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        processing_class=tokenizer,

    )

    trainer.train()

    adapter_dir = os.path.join(output_dir, "adapter")
    model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)
    print(f"Saved DPO adapter to: {adapter_dir}")

train_DPO()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/800 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss
20,0.657282,0.634701
40,0.631432,0.613973
60,0.630468,0.601946
80,0.627318,0.596354
100,0.595606,0.596220
120,0.582480,0.597505
140,0.549419,0.601243
160,0.573041,0.602112
180,0.585360,0.602139
200,0.604568,0.601955


Saved DPO adapter to: trl_dpo/adapter


In [11]:
@torch.inference_mode()
def infer(
    samples,
    adapter_dir: str,
    max_new_tokens: int = 256,
):
    tok = AutoTokenizer.from_pretrained(adapter_dir)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"

    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        device_map="auto",
        cache_dir=CACHE_DIR
    )

    model = PeftModel.from_pretrained(
        base,
        adapter_dir,
    )
    model.eval()
    model.config.use_cache = True
    device = next(model.parameters()).device
    outputs = []

    for item in samples:

        messages = [
            {
                "role": "user",
                "content": item["question"],
            }
        ]

        prompt = tok.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = tok(
            prompt,
            return_tensors="pt",
        ).to(device)

        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
        )

        new_tokens = generated_ids[0][inputs["input_ids"].shape[1]:]
        answer = tok.decode(
            new_tokens,
            skip_special_tokens=True,
        ).strip()
        outputs.append(answer)

    return outputs

In [12]:
samples = [
    {"question": "Explain overfitting simply."},
]

outs = infer(
    samples,
    adapter_dir="trl_dpo/adapter",
)

print(outs[0])

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Overfitting is a common problem in machine learning where a model learns the training data too well and performs exceptionally well on that specific dataset but poorly generalizes to new, unseen data.

Imagine you're trying to predict whether it will rain tomorrow based on historical weather patterns. If your model is trained using only past rainy days without considering other factors like temperature or humidity, it might perform very well on those rainy days but struggle when predicting non-rainy days. This inability to generalize to new situations is an example of overfitting.

To avoid overfitting, we can use techniques such as regularization (adding penalties for complex models), cross-validation (evaluating performance across different subsets of data), and feature selection (choosing relevant features). These methods help ensure our model doesn't become overly specialized to the training data and instead becomes more robust and adaptable to new inputs.
